# About this notebook (chipathon-2026-gf180mcu-padring)

This notebook is **historical / pedagogical**. It shows how the
`workshop` slot in this repository was *originally constructed*
from the upstream `wafer-space/gf180mcu-project-template` plus
Juan Moya's `padring_gf180` reference.

**You do not need to run this notebook to use the fork.** The
fork you are reading already contains the six files that define
the workshop slot (see `README.md` and `docs/workshop-slot-spec.md`).
To build the slot, run:

```bash
scripts/run_native.sh          # nix-shell build
# or
scripts/run_docker_iic.sh      # inspection inside iic-osic-tools
```

The notebook below was written in the context of the wider
[`eda-agents`](https://github.com/Mauricio-xx/eda-agents) tutorial
series (`tutorials/rtl2gds-gf180-docker`) before this fork existed
as a standalone repo. It uses a Docker-centric path
(`hpretl/iic-osic-tools:next`, bind-mount at `/foss/designs`) and
host paths under `~/eda/designs/chipathon_padring/`. If you want to
re-run it, create those directories first; otherwise read it as a
walkthrough of the slot-creation process.

**Credits** for the pad layout: Juan Moya
(https://github.com/JuanMoya/padring_gf180). Template author:
Leo Moser / wafer-space
(https://github.com/wafer-space/gf180mcu-project-template).
See `CREDITS.md` for per-artifact attribution.


# Chipathon 2026 workshop padring — building a custom slot

Walk-through of **creating a new `slot_*.yaml`** in the wafer-space
`gf180mcu-project-template`, mirroring the padring that JuanMoya
published at https://github.com/JuanMoya/padring_gf180. The
reference uses the stand-alone `padring` tool; this notebook produces
the same padring topology (88 pads + 4 corners on a 2935x2935 um
die) natively inside LibreLane's full-chip flow.

**Pedagogical goal.** Show that a *slot* is just:

1. One block in `src/slot_defines.svh` setting four pad counts
   (`NUM_DVDD_PADS`, `NUM_DVSS_PADS`, `NUM_BIDIR_PADS`,
   `NUM_ANALOG_PADS`).
2. One YAML file `librelane/slots/slot_<name>.yaml` listing the
   pad instance names in clockwise order from the SW corner.
3. One line in `Makefile` registering the slot name.

No code changes to `chip_top.sv` are needed if the template's
generate loops already produce the right number of pad instances;
we only tweak `chip_core.sv` (optional) so the core is a trivial
counter instead of the SRAM-based default, and strip the SRAM
macros from `config.yaml`.

**Not self-contained.** Needs a clone of the template + the
wafer-space GF180MCU PDK fork (several hundred MB). The notebook
clones them into `~/eda/designs/chipathon_padring/` on the host
if you flip `RUN_CLONE_TEMPLATE = True`.

**Runtime for the flow itself:** 35–45 min on a modern laptop.
All `RUN_*` flags default to `False` so the first pass rehearses
the commands without executing anything heavy.

**Target padring** (from the JuanMoya reference):

| side | pads                                                       |
|------|------------------------------------------------------------|
| N    | 16 analog + 2 power (vdd_ana1/vss_ana1) + 4 bidir          |
| E    | 12 analog + 2 power (vss_dig2/vdd_dig1) + 8 bidir          |
| S    | 20 analog + 2 power (vdd_ana2/vss_ana2) + clk_pad + rst_n  |
| W    | 12 analog + 2 power (vdd_dig4/vss_dig3) + 8 bidir          |

Totals: 60 analog, 20 bidir, 4 dvdd, 4 dvss, plus the two
hardcoded single-instance inputs (`clk_pad`, `rst_n_pad`) from the
stock template.

## Step 0 — configuration

In [ ]:
from pathlib import Path
import os
import subprocess
import textwrap

# ---- flags ----
RUN_CLONE_TEMPLATE  = False   # ~250 MB, ~30 s
RUN_CLONE_PDK_FORK  = False   # ~500 MB, 1-2 min
RUN_PATCH_SOURCES   = False   # text patches on 4 files (idempotent-ish)
RUN_FLOW            = False   # 35-45 min, ~18 GB runs dir

# ---- container ----
CONTAINER_NAME = "gf180"

# ---- host paths ----
HOST_WORKSPACE   = Path.home() / "eda" / "designs"
HOST_PROJECT_DIR = HOST_WORKSPACE / "chipathon_padring"
HOST_TEMPLATE    = HOST_PROJECT_DIR / "template"

# ---- container paths (bind-mount: ~/eda/designs <-> /foss/designs) ----
CONTAINER_PROJECT  = "/foss/designs/chipathon_padring"
CONTAINER_TEMPLATE = f"{CONTAINER_PROJECT}/template"

# ---- GF180MCU identifiers ----
PDK_NAME     = "gf180mcuD"
STD_CELL_LIB = "gf180mcu_fd_sc_mcu7t5v0"

TEMPLATE_REPO = "https://github.com/wafer-space/gf180mcu-project-template.git"
PDK_FORK_REPO = "https://github.com/wafer-space/gf180mcu.git"
PDK_FORK_TAG  = "1.8.0"


def run_or_print(cmd, do_it, *, shell_on_container=False, timeout=None, cwd=None):
    if shell_on_container:
        print(f"$ docker exec {CONTAINER_NAME} bash -lc '<script>'")
        print(textwrap.indent(cmd, "  | "))
    else:
        print("$ " + " ".join(cmd) + (f"   (cwd={cwd})" if cwd else ""))
    if not do_it:
        print("  (skipped -- flip the RUN_* flag to execute)\n")
        return None
    print("  (executing...)")
    args = (
        ["docker", "exec", CONTAINER_NAME, "bash", "-lc", cmd]
        if shell_on_container else cmd
    )
    proc = subprocess.run(args, capture_output=True, text=True, timeout=timeout, cwd=cwd)
    if proc.stdout.strip():
        print(proc.stdout[-4000:])
    if proc.returncode != 0 and proc.stderr.strip():
        print("STDERR (tail):")
        print(proc.stderr[-2000:])
    print(f"  returncode={proc.returncode}\n")
    return proc


HOST_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project dir:       {HOST_PROJECT_DIR}")
print(f"Template (host):   {HOST_TEMPLATE}")
print(f"Template (ctr):    {CONTAINER_TEMPLATE}")

## Step 1 — prerequisites

In [ ]:
def ok(label, cond, detail=""):
    print(f"{'OK ' if cond else '!! '}{label}" + (f"  -- {detail}" if detail else ""))
    return cond

good = True
good &= ok("Docker image hpretl/iic-osic-tools:next present",
           subprocess.run(["docker", "image", "inspect", "hpretl/iic-osic-tools:next"],
                          capture_output=True).returncode == 0)
good &= ok(f"Container '{CONTAINER_NAME}' running",
           CONTAINER_NAME in subprocess.run(
               ["docker", "ps", "--filter", f"name={CONTAINER_NAME}", "--format", "{{.Names}}"],
               capture_output=True, text=True).stdout)

## Step 2 — clone template + wafer-space PDK fork

Two clones into `~/eda/designs/chipathon_padring/`:

1. `gf180mcu-project-template` — the full-chip skeleton
   (`chip_top.sv`, `chip_core.sv`, `librelane/config.yaml`, `Makefile`,
   existing `slots/slot_{1x1,0p5x1,1x0p5,0p5x0p5}.yaml`).
2. The wafer-space GF180MCU PDK fork (tag `1.8.0`). Lives inside the
   template at `template/gf180mcu/`.

In [ ]:
# 2a -- template
if HOST_TEMPLATE.exists():
    print(f"Template already at {HOST_TEMPLATE}  (skipping clone)")
else:
    run_or_print(
        ["git", "clone", "--depth", "1", TEMPLATE_REPO, str(HOST_TEMPLATE)],
        RUN_CLONE_TEMPLATE,
    )

# 2b -- wafer-space PDK fork
pdk_fork = HOST_TEMPLATE / "gf180mcu"
if pdk_fork.exists():
    print(f"Wafer-space PDK fork already at {pdk_fork}  (skipping clone)")
else:
    run_or_print(
        ["git", "clone", "--depth", "1", "--branch", PDK_FORK_TAG,
         PDK_FORK_REPO, str(pdk_fork)],
        RUN_CLONE_PDK_FORK,
    )

## Step 3 — patch `src/slot_defines.svh`

Add a `SLOT_WORKSHOP` block to the pad-count header. The numbers
drive the generate loops in `chip_top.sv`:

```verilog
`ifdef SLOT_WORKSHOP
`define NUM_DVDD_PADS 4
`define NUM_DVSS_PADS 4
`define NUM_INPUT_PADS 0
`define NUM_BIDIR_PADS 20
`define NUM_ANALOG_PADS 60
`endif
```

In [ ]:
slot_defines = HOST_TEMPLATE / "src" / "slot_defines.svh"

SLOT_WORKSHOP_BLOCK = """
// Chipathon 2026 workshop padring -- 2935x2935 um die.
// Mirror of JuanMoya's padring_gf180/workshop_padring.cfg:
//   60 analog (asig_5p0), 20 bidir (bi_24t on this template), 4 dvdd
//   + 4 dvss, 4 corners.  clk_pad and rst_n_pad remain the stock
//   single-instance inputs (listed as literals in slot_workshop.yaml,
//   they do not count toward NUM_INPUT_PADS).
`ifdef SLOT_WORKSHOP

// Power/ground pads for core and I/O
`define NUM_DVDD_PADS 4
`define NUM_DVSS_PADS 4

// Signal pads
`define NUM_INPUT_PADS 0
`define NUM_BIDIR_PADS 20
`define NUM_ANALOG_PADS 60

`endif
"""

if not slot_defines.exists():
    print(f"{slot_defines} not found. Clone template first (Step 2a).")
else:
    src = slot_defines.read_text()
    already = "SLOT_WORKSHOP" in src
    print(f"SLOT_WORKSHOP block present: {already}")
    if RUN_PATCH_SOURCES and not already:
        slot_defines.write_text(src.rstrip() + "\n" + SLOT_WORKSHOP_BLOCK)
        print(f"-> appended workshop block to {slot_defines}")
    elif not RUN_PATCH_SOURCES:
        print("\n(Dry run. Flip RUN_PATCH_SOURCES = True to apply.)")
        print(textwrap.indent(SLOT_WORKSHOP_BLOCK.strip(), "    "))

## Step 4 — write `librelane/slots/slot_workshop.yaml`

The 88 pad instances go into the four `PAD_*` lists **in clockwise
order from the SW corner**:

- `PAD_SOUTH`: W → E (24 pads: clk_pad, rst_n_pad, 10 analog, 2
  power, 10 analog).
- `PAD_EAST`:  S → N (22 pads: 10 analog, 2 power, 8 bidir, 2 analog).
- `PAD_NORTH`: E → W (22 pads: **reversed** relative to JuanMoya's
  north listing; 4 bidir, 6 analog, 2 power, 10 analog).
- `PAD_WEST`:  N → S (22 pads: **reversed** relative to JuanMoya's
  west listing; 2 analog, 8 bidir, 2 power, 10 analog).

Index mapping into the generate loops of `chip_top.sv`:

| label                        | instance                  |
|------------------------------|---------------------------|
| `anaN` (N=1..60)             | `analog[N-1].pad`         |
| `configN` (N=1..20)          | `bidir[N-1].pad`          |
| `vdd_ana1 / vdd_dig1 / vdd_ana2 / vdd_dig4` | `dvdd_pads[0..3].pad` |
| `vss_ana1 / vss_dig2 / vss_ana2 / vss_dig3` | `dvss_pads[0..3].pad` |
| `clk_pad / rst_n_pad` (inputs) | literal single instances |

In [ ]:
slot_yaml = HOST_TEMPLATE / "librelane" / "slots" / "slot_workshop.yaml"

SLOT_WORKSHOP_YAML = textwrap.dedent('''\
    # Chipathon 2026 workshop padring slot.
    # Mirrors https://github.com/JuanMoya/padring_gf180 for use with
    # LibreLane's full-chip flow instead of the stand-alone padring tool.

    FP_SIZING: absolute
    DIE_AREA:  [0, 0, 2935, 2935]
    CORE_AREA: [442, 442, 2493, 2493]

    VERILOG_DEFINES: ["SLOT_WORKSHOP"]

    # ----- South: SW -> SE (24 pads) -----
    PAD_SOUTH: [
        clk_pad, rst_n_pad,
        "analog\\\\[28\\\\].pad", "analog\\\\[29\\\\].pad",
        "analog\\\\[30\\\\].pad", "analog\\\\[31\\\\].pad",
        "analog\\\\[32\\\\].pad", "analog\\\\[33\\\\].pad",
        "analog\\\\[34\\\\].pad", "analog\\\\[35\\\\].pad",
        "analog\\\\[36\\\\].pad", "analog\\\\[37\\\\].pad",
        "dvdd_pads\\\\[2\\\\].pad", "dvss_pads\\\\[2\\\\].pad",
        "analog\\\\[38\\\\].pad", "analog\\\\[39\\\\].pad",
        "analog\\\\[40\\\\].pad", "analog\\\\[41\\\\].pad",
        "analog\\\\[42\\\\].pad", "analog\\\\[43\\\\].pad",
        "analog\\\\[44\\\\].pad", "analog\\\\[45\\\\].pad",
        "analog\\\\[46\\\\].pad", "analog\\\\[47\\\\].pad",
    ]

    # ----- East: SE -> NE (22 pads) -----
    PAD_EAST: [
        "analog\\\\[27\\\\].pad", "analog\\\\[26\\\\].pad",
        "analog\\\\[25\\\\].pad", "analog\\\\[24\\\\].pad",
        "analog\\\\[23\\\\].pad", "analog\\\\[22\\\\].pad",
        "analog\\\\[21\\\\].pad", "analog\\\\[20\\\\].pad",
        "analog\\\\[19\\\\].pad", "analog\\\\[18\\\\].pad",
        "dvss_pads\\\\[1\\\\].pad", "dvdd_pads\\\\[1\\\\].pad",
        "bidir\\\\[11\\\\].pad", "bidir\\\\[10\\\\].pad",
        "bidir\\\\[9\\\\].pad",  "bidir\\\\[8\\\\].pad",
        "bidir\\\\[7\\\\].pad",  "bidir\\\\[6\\\\].pad",
        "bidir\\\\[5\\\\].pad",  "bidir\\\\[4\\\\].pad",
        "analog\\\\[17\\\\].pad", "analog\\\\[16\\\\].pad",
    ]

    # ----- North: NE -> NW (22 pads; reversed vs JuanMoya) -----
    PAD_NORTH: [
        "bidir\\\\[3\\\\].pad", "bidir\\\\[2\\\\].pad",
        "bidir\\\\[1\\\\].pad", "bidir\\\\[0\\\\].pad",
        "analog\\\\[15\\\\].pad", "analog\\\\[14\\\\].pad",
        "analog\\\\[13\\\\].pad", "analog\\\\[12\\\\].pad",
        "analog\\\\[11\\\\].pad", "analog\\\\[10\\\\].pad",
        "dvss_pads\\\\[0\\\\].pad", "dvdd_pads\\\\[0\\\\].pad",
        "analog\\\\[9\\\\].pad", "analog\\\\[8\\\\].pad",
        "analog\\\\[7\\\\].pad", "analog\\\\[6\\\\].pad",
        "analog\\\\[5\\\\].pad", "analog\\\\[4\\\\].pad",
        "analog\\\\[3\\\\].pad", "analog\\\\[2\\\\].pad",
        "analog\\\\[1\\\\].pad", "analog\\\\[0\\\\].pad",
    ]

    # ----- West: NW -> SW (22 pads; reversed vs JuanMoya) -----
    PAD_WEST: [
        "analog\\\\[59\\\\].pad", "analog\\\\[58\\\\].pad",
        "bidir\\\\[12\\\\].pad", "bidir\\\\[13\\\\].pad",
        "bidir\\\\[14\\\\].pad", "bidir\\\\[15\\\\].pad",
        "bidir\\\\[16\\\\].pad", "bidir\\\\[17\\\\].pad",
        "bidir\\\\[18\\\\].pad", "bidir\\\\[19\\\\].pad",
        "dvdd_pads\\\\[3\\\\].pad", "dvss_pads\\\\[3\\\\].pad",
        "analog\\\\[57\\\\].pad", "analog\\\\[56\\\\].pad",
        "analog\\\\[55\\\\].pad", "analog\\\\[54\\\\].pad",
        "analog\\\\[53\\\\].pad", "analog\\\\[52\\\\].pad",
        "analog\\\\[51\\\\].pad", "analog\\\\[50\\\\].pad",
        "analog\\\\[49\\\\].pad", "analog\\\\[48\\\\].pad",
    ]
    ''')

# Substitute the escape placeholders (yaml needs single backslashes,
# and Python strings need them doubled above).
SLOT_WORKSHOP_YAML = SLOT_WORKSHOP_YAML.replace('\\\\\\\\', '\\\\')

if not slot_yaml.parent.exists():
    print(f"{slot_yaml.parent} not found. Clone template first (Step 2a).")
elif RUN_PATCH_SOURCES:
    slot_yaml.write_text(SLOT_WORKSHOP_YAML)
    print(f"-> wrote {slot_yaml}")
else:
    print(f"Would write {slot_yaml}:\n")
    print(textwrap.indent(SLOT_WORKSHOP_YAML[:1500] + "  ...", "    "))

## Step 5 — patch `Makefile`, `chip_core.sv`, `config.yaml`, `pdn_cfg.tcl`

Four small edits:

1. **Makefile**: register `workshop` in `AVAILABLE_SLOTS` so `make
   librelane SLOT=workshop` is accepted.
2. **`src/chip_core.sv`**: replace the SRAM-based default with a
   trivial 20-bit counter. Required because the stock `chip_core`
   instantiates SRAMs we no longer place.
3. **`librelane/config.yaml`**: drop the SRAM `MACROS` entry + the
   corresponding `PDN_MACRO_CONNECTIONS` lines.
4. **`librelane/pdn_cfg.tcl`**: drop the two SRAM-specific
   `define_pdn_grid` blocks.

See the cell below for exact diffs; they are applied when
`RUN_PATCH_SOURCES = True`.

In [ ]:
# 5a -- Makefile: register the slot
mk = HOST_TEMPLATE / "Makefile"
MK_OLD = "AVAILABLE_SLOTS = 1x1 0p5x1 1x0p5 0p5x0p5"
MK_NEW = "AVAILABLE_SLOTS = 1x1 0p5x1 1x0p5 0p5x0p5 workshop"
if mk.exists():
    src = mk.read_text()
    print(f"Makefile already has workshop slot: {'workshop' in src}")
    if RUN_PATCH_SOURCES and MK_OLD in src:
        mk.write_text(src.replace(MK_OLD, MK_NEW))
        print("-> patched Makefile")

# 5b -- replace chip_core.sv
CHIP_CORE_WORKSHOP = '''\
// Minimal chip_core for the Chipathon 2026 workshop padring slot.
// A free-running counter drives the 20 bidir pads; the 60 analog
// pads stay unconnected at the core level (downstream designs can
// wire them to custom IP).

`default_nettype none

module chip_core #(
    parameter NUM_INPUT_PADS,
    parameter NUM_BIDIR_PADS,
    parameter NUM_ANALOG_PADS
    )(
    `ifdef USE_POWER_PINS
    inout  wire VDD,
    inout  wire VSS,
    `endif

    input  wire clk,
    input  wire rst_n,

    input  wire [NUM_INPUT_PADS-1:0] input_in,
    output wire [NUM_INPUT_PADS-1:0] input_pu,
    output wire [NUM_INPUT_PADS-1:0] input_pd,

    input  wire [NUM_BIDIR_PADS-1:0] bidir_in,
    output wire [NUM_BIDIR_PADS-1:0] bidir_out,
    output wire [NUM_BIDIR_PADS-1:0] bidir_oe,
    output wire [NUM_BIDIR_PADS-1:0] bidir_cs,
    output wire [NUM_BIDIR_PADS-1:0] bidir_sl,
    output wire [NUM_BIDIR_PADS-1:0] bidir_ie,
    output wire [NUM_BIDIR_PADS-1:0] bidir_pu,
    output wire [NUM_BIDIR_PADS-1:0] bidir_pd,

    inout  wire [NUM_ANALOG_PADS-1:0] analog
);
    assign input_pu = '0;
    assign input_pd = '0;
    assign bidir_oe = '1;
    assign bidir_cs = '0;
    assign bidir_sl = '0;
    assign bidir_ie = ~bidir_oe;
    assign bidir_pu = '0;
    assign bidir_pd = '0;

    logic _unused;
    assign _unused = &{1'b0, bidir_in, input_in};

    logic [NUM_BIDIR_PADS-1:0] count;
    always_ff @(posedge clk) begin
        if (!rst_n) count <= '0;
        else        count <= count + 1;
    end
    assign bidir_out = count;
endmodule

`default_nettype wire
'''

chip_core = HOST_TEMPLATE / "src" / "chip_core.sv"
if chip_core.exists():
    already = "Chipathon 2026" in chip_core.read_text()
    print(f"chip_core.sv already workshop-patched: {already}")
    if RUN_PATCH_SOURCES and not already:
        chip_core.write_text(CHIP_CORE_WORKSHOP)
        print("-> replaced chip_core.sv")

# 5c/5d -- strip SRAM block from config.yaml and pdn_cfg.tcl.
# For brevity the full diffs are applied via the companion script
# in the /foss/designs/chipathon_padring/template directory on the
# reference run; in a clean clone the user should edit these two
# files by hand (or run the cell below which does both in one shot).
cfg = HOST_TEMPLATE / "librelane" / "config.yaml"
pdn = HOST_TEMPLATE / "librelane" / "pdn_cfg.tcl"

if RUN_PATCH_SOURCES:
    if cfg.exists():
        s = cfg.read_text()
        # Delete from the sram MACROS entry header up to the end of
        # PDN_MACRO_CONNECTIONS, replacing with a minimal stub.
        start_marker = "  gf180mcu_fd_ip_sram__sram512x8m8wm1:"
        end_marker   = 'PDN_MACRO_CONNECTIONS:\n- "i_chip_core.sram_1 VDD VSS VDD VSS"'
        if start_marker in s and end_marker in s:
            before, rest = s.split(start_marker, 1)
            _, after = rest.split(end_marker, 1)
            s = before.rstrip() + "\n\n" + "PDN_MACRO_CONNECTIONS: []" + after
            cfg.write_text(s)
            print("-> stripped SRAM block from config.yaml")
    if pdn.exists():
        s = pdn.read_text()
        if "i_chip_core.sram_0" in s:
            lines = s.splitlines(keepends=True)
            out, skip = [], False
            for ln in lines:
                if "# SRAM macros" in ln:
                    skip = True
                if skip:
                    continue
                out.append(ln)
            pdn.write_text("".join(out))
            print("-> stripped SRAM PDN blocks from pdn_cfg.tcl")
else:
    print("(Dry run; flip RUN_PATCH_SOURCES = True to apply config.yaml + pdn_cfg.tcl edits.)")

## Step 6 — run the flow

Same invocation as the stock template, but with `SLOT=workshop`.

In [ ]:
flow_script = textwrap.dedent(f"""
    set -e
    cd {CONTAINER_TEMPLATE}
    source sak-pdk-script.sh {PDK_NAME} {STD_CELL_LIB}
    make librelane \\
        SLOT=workshop \\
        PDK={PDK_NAME} \\
        PDK_ROOT={CONTAINER_TEMPLATE}/gf180mcu
""").strip()

run_or_print(flow_script, RUN_FLOW, shell_on_container=True, timeout=4000)

## Step 7 — display the workshop padring render

In [ ]:
from IPython.display import Image, display

png_path = HOST_TEMPLATE / "final" / "render" / "chip_top.png"
gds_path = HOST_TEMPLATE / "final" / "gds"    / "chip_top.gds"

if png_path.exists():
    print(f"Render: {png_path}")
    display(Image(str(png_path)))
else:
    print(f"Render PNG not found: {png_path}")
    if gds_path.exists():
        print(f"GDS is there: {gds_path}")
        print(f"  open it with:  klayout {gds_path}")
    else:
        print("No GDS either -- Step 6 has not produced artifacts yet.")

## Cleanup and follow-ups

```bash
rm -rf ~/eda/designs/chipathon_padring/template/librelane/runs
docker stop gf180
```

### Next steps for chipathon participants

1. **Adjust the die**. `DIE_AREA` in the slot yaml is the single
   knob; make it smaller if your analog circuitry needs less room.
2. **Rebalance the pad count**. Drop / add analog pads by editing
   `NUM_ANALOG_PADS` in `slot_defines.svh` and the corresponding
   index range in the `PAD_*` lists.
3. **Wire real analog macros** into `chip_core.sv`. Each
   `analog[i]` port becomes the `ASIG5V` pin of one `asig_5p0`
   pad; route each to your analog block.
4. **Swap `bi_24t` for `bi_t`** if you want bit-exact match with
   the JuanMoya reference (edit `chip_top.sv` bidir generate loop).
5. **Run a cocotb testbench** (`make verify SLOT=workshop`) to
   sanity-check connectivity before the full flow.